<a href="https://colab.research.google.com/github/MuhammadAjlal2004/A-Quantitative-Greenwashing-Index-for-the-Luxury-Sector/blob/main/SW(Data_Extraction).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Note: Data extraction utilized ProQuest TDM Studio's internal server architecture. The code blocks below are preserved for methodological transparency but cannot be executed in a local or standard cloud environment.

In [ ]:
# Extracting XML Documents

import os
import pandas as pd
from bs4 import BeautifulSoup
import sys

folder_path = './data/FinalSWDataset2'

# Creating an empty list. This will act as a bucket to hold the text of each article as we extract it:
articles_data = []

print(f"Scanning folder: {folder_path}...")

try:
    # os.listdir gets a list of everything in the folder.
    all_files = os.listdir(folder_path)
    # A "list comprehension" is used to filter the list so it only keeps files ending in '.xml'
    xml_files = [f for f in all_files if f.endswith('.xml')]
    print(f"Found {len(xml_files)} XML files. Extracting text...")

    # Extracting the text of each XML file
    for filename in xml_files:
        file_path = os.path.join(folder_path, filename)

        # Opening the file in 'read' mode ('r'). 'utf-8' encoding prevents errors with special characters (like accents in Hermès)
        with open(file_path, 'r', encoding='utf-8') as file:
            # Beautiful soup parses through the raw XML code
            soup = BeautifulSoup(file, 'html.parser')
            clean_text = soup.get_text(separator=' ', strip=True)
            articles_data.append({'text': clean_text})

    # Converting the list into a dataframe
    df = pd.DataFrame(articles_data)
    df['text'] = df['text'].astype(str).str.lower()

    print(f"{len(df)} documents compiled into a single dataset")

except FileNotFoundError:
    print(f"Cannot find the folder {folder_path}")
    sys.exit()

In [ ]:
# Creating the dictionaries for text processing (through the NLP toolkit) and context-window LDA Topic Modeling to filter out thematic noise across the XML documents.

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# The parent-child mapping dictionary (this dictionary maps subsidiary aliases [keys] to their ultimate parent conglomerate [values])
brand_hierarchy = {
    # LVMH
    'lvmh': 'LVMH', 'louis vuitton': 'LVMH', 'dior': 'LVMH', 'fendi': 'LVMH', 'celine': 'LVMH',
    'givenchy': 'LVMH', 'loewe': 'LVMH', 'kenzo': 'LVMH', 'bulgari': 'LVMH', 'bvlgari': 'LVMH',
    'tag heuer': 'LVMH', 'hublot': 'LVMH', 'pucci': 'LVMH', 'chaumet': 'LVMH',

    # Kering
    'kering': 'Kering', 'gucci': 'Kering', 'saint laurent': 'Kering', 'balenciaga': 'Kering',
    'bottega veneta': 'Kering', 'alexander mcqueen': 'Kering', 'boucheron': 'Kering',

    # Richemont
    'richemont': 'Richemont', 'cartier': 'Richemont', 'chloe': 'Richemont', 'montblanc': 'Richemont',
    'piaget': 'Richemont', 'alaia': 'Richemont', 'delvaux': 'Richemont',

    # Capri Holdings
    'capri': 'Capri Holdings', 'versace': 'Capri Holdings', 'michael kors': 'Capri Holdings', 'jimmy choo': 'Capri Holdings',

    # Tapestry
    'tapestry': 'Tapestry', 'coach': 'Tapestry', 'kate spade': 'Tapestry',

    # Prada Group
    'prada': 'Prada Group', 'miu miu': 'Prada Group',

    # PVH Corp
    'calvin klein': 'PVH Corp', 'tommy hilfiger': 'PVH Corp',

    # OTB Group
    'diesel': 'OTB Group', 'marni': 'OTB Group', 'maison margiela': 'OTB Group', 'jil sander': 'OTB Group',

    # Puig
    'paco rabanne': 'Puig', 'nina ricci': 'Puig', 'carolina herrera': 'Puig',

    # Swatch Group
    'longines': 'Swatch Group', 'tissot': 'Swatch Group'
}

# Independent brands
independents = [
    'hermes', 'burberry', 'moncler', 'chanel', 'rolex', 'ferragamo', 'ralph lauren', 'hugo boss',
    'armani', 'valentino', 'dolce & gabbana', 'zegna', 'guess', 'puma', 'lacoste', 'vans',
    'patek philippe', 'audemars piguet', 'chopard', 'breitling', 'stella mccartney', 'jacquemus'
]

# This loops through the independents, capitalizes the first letter, and adds them to the dictionary
for brand in independents:
    brand_hierarchy[brand] = brand.title()

# Hierarchial brand detection
def get_parent_companies(text):
    parents_found = set() # Using a set ensures we don't count LVMH twice if both Dior and Fendi are in one article
    for search_term, parent_company in brand_hierarchy.items(): # Iterating through the dictionary. If the lowercase brand alias is in the text, add the Parent to the set
        if search_term in text:
            parents_found.add(parent_company)
    return list(parents_found) if parents_found else ['None']

# Applying the mapping to the XML text
df['Parent_Companies'] = df['text'].apply(get_parent_companies)

# Exploding so each Parent Company gets its own row for counting
df_exploded = df.explode('Parent_Companies')
df_exploded = df_exploded[df_exploded['Parent_Companies'] != 'None'] # Removing articles with no brand names

# Calculating the counts and creating an Index for the talk vs walk ratio:
climate_terms = ['sustainability', 'climate change', 'emissions', 'net-zero', 'carbon footprint', 'eco-friendly']
greenwash_terms = ['greenwash', 'greenwashing', 'misleading', 'unsubstantiated', 'deceptive', 'false claim', 'greenhush']

# Using lambda functions to count how many times the terms above appear in each article
df_exploded['C_Count'] = df_exploded['text'].apply(lambda x: sum(x.count(t) for t in climate_terms))
df_exploded['G_Count'] = df_exploded['text'].apply(lambda x: sum(x.count(t) for t in greenwash_terms))

# Group by the newly mapped Parent Companies:
brand_stats = df_exploded.groupby('Parent_Companies')[['C_Count', 'G_Count']].sum().reset_index()

# Calculating the Greenwashing Ratio. We add 0.0001 to prevent a "Divide by Zero" math error if C_Count is 0
brand_stats['Greenwash_Ratio'] = brand_stats['G_Count'] / (brand_stats['C_Count'] + 0.0001)

# LDA TOPIC MODELING:
from sklearn.feature_extraction import text # To grab the massive built-in stopword list

print("\n--- Generating Deep & Concise LDA Topics ---")

gw_sentences = []

# Instead of feeding the whole article (which causes noise), we split the article by periods ('.'). We only save the exact sentences that contain a greenwashing keyword
for doc_text in df_exploded[df_exploded['G_Count'] > 0]['text']:
    sentences = doc_text.split('.')
    for sentence in sentences:
        if any(term in sentence for term in greenwash_terms):
            gw_sentences.append(sentence.strip())

# Combining SCIKIT-LEARN'S built-in library with the junk words (this automatically catches 'because', 'our', 'whether', 'that', 'from', etc):
built_in_stopwords = list(text.ENGLISH_STOP_WORDS)

junk_words = [
    'january', 'february', 'march', 'april', 'may', 'june', 'july', 'jul', 'august',
    'september', 'october', 'november', 'december', 'monday', 'tuesday', 'wednesday',
    'thursday', 'friday', 'saturday', 'sunday', 'united', 'newcastle', 'asia', 'pacific',
    'said', 'brand', 'fashion', 'company', 'luxury', 'dr', 'school', 'health', 'uc',
    'professor', 'students', 'canterbury', 'guess', 'traded', 'upon', 'email', 'feedback',
    'please', 'financials', 'page', 'cover', 'win', 'fm', 'club', 'nfl', 'race', 'racing',
    'galway', 'airtrunk', 'sq', 'cooling', 'faro', 'jpg', 'telco', 'realty', 'kt', 'centers', 'esg',
    'datacenterdynamics', 'power', 'website', 'report', 'research', 'university',
    'http', 'https', 'copyright', '2021', '2022', '2023', '2024'
]

deep_stopwords = built_in_stopwords + junk_words

# Running the LDA model
if len(gw_sentences) > 0:
    # CountVectorizer turns the sentences into a mathematical matrix of word counts
    vectorizer = CountVectorizer(
        max_df=0.5,
        min_df=2,
        stop_words=deep_stopwords,
        # token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b' is a regex that forces the model to ignore numbers and punctuation, keeping ONLY words 2 letters or longer
        token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b'
    )
    # Fitting the text to the matrix
    dtm = vectorizer.fit_transform(gw_sentences)
    # Initializing the LDA algorithm, forcing it to find exactly 4 underlying topics (n_components=4)
    lda = LatentDirichletAllocation(n_components=4, random_state=42)
    lda.fit(dtm)
    # Loop through the 4 created topics and print the top 10 most heavily weighted words for each
    for idx, topic in enumerate(lda.components_):
        print(f"Deep Theme {idx+1}: ", [vectorizer.get_feature_names_out()[i] for i in topic.argsort()[-10:]])

In [ ]:
display(brand_stats) #Displays the G_Count (Greenwashing terms) and C_Counts (Climate terms) for the talk vs walk ratio for luxury brands